# Procesamiento de Tarifas (TMEC / USMCA)
## Fases de Tratado
### Fase 0: Configuración General

El objetivo de este script es procesar la base de datos arancelaria general para filtrar y calcular exclusivamente las tarifas preferenciales aplicables bajo el tratado TMEC (USMCA). Transforma las tasas crudas (ad valorem, específicas y otras) en una fórmula final legible basada en los códigos indicadores del tratado.

Dependencias requeridas:
- `pandas (pd):` Manipulación de datos, filtrado y aplicación vectorizada de lógicas.
- `os:` Manejo de rutas y creación de directorios del sistema.

Variables Globales:
- **Rutas:** Ubicaciones de los inputs crudos (Tariff Database 2025) y la ruta de salida del archivo procesado.

In [1]:
import pandas as pd
import os

# --- RUTAS DE ARCHIVOS ---
PATH_INPUT_TMEC = "../data/raw/tariff_database_2025.xlsx"
PATH_OUTPUT_TMEC = "../data/intermediate/fracciones_tmec.xlsx"

print("--- CONFIGURACIÓN CARGADA ---")
print(f"Input:  {PATH_INPUT_TMEC}")
print(f"Output: {PATH_OUTPUT_TMEC}")

### Fase 0.5: Definición de Funciones

Se desanidan las lógicas de formato y cálculo para facilitar la trazabilidad:

**1. Funciones Auxiliares (`_nombre`)**
Herramientas de formateo a nivel de celda individual.

- **`_es_tasa_compleja`**: Identifica el marcador de sistema "9999.99" que indica que la tarifa no puede ser calculada de forma simple.
- **`_formatear_moneda`**: Convierte valores decimales crudos en cadenas con formato de moneda (ej. "$0.5").
- **`_formatear_porcentaje`**: Transforma decimales crudos a porcentajes legibles sin decimales excesivos (ej. "0.05" -> "5%").
- **`_construir_parte_especifica`**: Ensambla el bloque matemático de la tarifa sumando el multiplicador de unidad correspondiente (ej. "$0.5 * KG").

**2. Funciones Intermedias (`__nombre__`)**
Lógica estructural que procesa filas completas.

- **`__calcular_final_rate__`**: Lee los metadatos de una fila completa, invoca a las funciones auxiliares de formato y, a través de un árbol de decisión basado en el `usmca_rate_type_code` (0 al 9, K, X, T), construye la fórmula arancelaria final legible.

**3. Funciones Principales (`nombre`)**
Orquestadores de procesos de negocio.

- **`procesar_tarifas_tmec`**: Carga el Excel, selecciona las columnas de interés, filtra exclusivamente las filas marcadas con los indicadores TMEC ('S' o 'S+'), y aplica la función de cálculo a todo el subconjunto.

In [2]:
# --- FUNCIONES AUXILIARES (Formato y Limpieza) ---

def _es_tasa_compleja(valor_str):
    """Detecta si el valor contiene el marcador de complejidad 9999.99."""
    return "9999.99" in str(valor_str)

def _formatear_moneda(valor):
    """
    Convierte un valor string (ej '0.5') a formato moneda ('$0.5').
    Si es vacío, devuelve '$0'.
    """
    if pd.isna(valor) or str(valor).strip() == "":
        return "$0"
    
    s_val = str(valor).strip()
    if _es_tasa_compleja(s_val):
        return "COMPLEX"
    
    try:
        return f"${s_val}"
    except:
        return f"${s_val}"

def _formatear_porcentaje(valor):
    """
    Convierte un valor string decimal (ej '0.05') a porcentaje ('5%').
    """
    if pd.isna(valor) or str(valor).strip() == "":
        return "0%"
    
    s_val = str(valor).strip()
    if _es_tasa_compleja(s_val):
        return "COMPLEX"
    
    try:
        # Convertir a float y multiplicar por 100
        float_val = float(s_val)
        porcentaje = float_val * 100
        # Usamos :g para evitar ceros decimales innecesarios (5.0 -> 5)
        return f"{porcentaje:g}%"
    except:
        return f"{s_val}"

def _construir_parte_especifica(tasa_formateada, unidad_codigo):
    """
    Maneja la lógica: Tasa * Unidad. Si la unidad es vacía asume Tasa/unidad.
    """
    if tasa_formateada == "COMPLEX":
        return "COMPLEX"
    
    unidad = str(unidad_codigo).strip() if pd.notna(unidad_codigo) else ""
    
    if unidad == "":
        return f"{tasa_formateada}/unidad"
    else:
        return f"{tasa_formateada} * {unidad}"


# --- FUNCIONES INTERMEDIAS (Lógica Estructural) ---

def __calcular_final_rate__(row):
    """
    Aplica el árbol de decisión tarifario según el usmca_rate_type_code.
    """
    code = str(row.get('usmca_rate_type_code', '')).strip()
    
    # Extracción de valores crudos
    raw_spec = row.get('usmca_specific_rate')
    raw_other = row.get('usmca_other_rate')
    raw_adval = row.get('usmca_ad_val_rate')
    q1 = row.get('quantity_1_code')
    q2 = row.get('quantity_2_code')

    # Transformación de formatos base
    spec_fmt = _formatear_moneda(raw_spec)      
    other_fmt = _formatear_moneda(raw_other)    
    adval_fmt = _formatear_porcentaje(raw_adval)

    # Validación temprana de tasas complejas indescifrables
    if "COMPLEX" in [spec_fmt, other_fmt, adval_fmt]:
        return "Tasa Compleja (Revisar HTS)"

    # Construcción de los bloques de fórmula
    bloque_spec_q1 = _construir_parte_especifica(spec_fmt, q1)
    bloque_spec_q2 = _construir_parte_especifica(spec_fmt, q2)
    bloque_other_q2 = _construir_parte_especifica(other_fmt, q2)
    bloque_adval = f"{adval_fmt} * Valor"

    # Árbol de decisiones tarifarias oficiales
    if code == '0':
        return "Free"
    elif code == '1':
        return bloque_spec_q1
    elif code == '2':
        return bloque_spec_q2
    elif code == '3':
        return f"({bloque_spec_q1}) + ({bloque_other_q2})"
    elif code == '4':
        return f"({bloque_spec_q1}) + ({bloque_adval})"
    elif code == '5':
        return f"({bloque_spec_q2}) + ({bloque_adval})"
    elif code == '6':
        return f"({bloque_spec_q1}) + ({bloque_other_q2}) + ({bloque_adval})"
    elif code == '7':
        return bloque_adval
    elif code == '9':
        return f"{adval_fmt} * Derived Duty (Refer to HTS)"
    elif code in ['K', 'X']:
        return "Refer to HTS for duty computation procedures"
    elif code == 'T':
        return "Compute at 10-digit level. Refer to HTS"
    else:
        return f"Unknown Code ({code})"


# --- FUNCIONES PRINCIPALES (Negocio) ---

def procesar_tarifas_tmec(ruta_input):
    """
    Orquestador maestro: Filtra la base por elegibilidad TMEC y aplica los cálculos.
    """
    print(f">> Cargando archivo de Excel base desde: {ruta_input}")
    try:
        # Leemos como string para no perder ceros a la izquierda
        df = pd.read_excel(ruta_input, dtype=str)
    except FileNotFoundError:
        print(f"❌ ERROR: No se encuentra el archivo en: {ruta_input}")
        return pd.DataFrame()
    except Exception as e:
        print(f"❌ ERROR inesperado al leer el archivo Excel: {e}")
        return pd.DataFrame()

    # Acotamos columnas por eficiencia
    columnas_interes = [
        "hts8", "quantity_1_code", "quantity_2_code", "col1_special_text", 
        "usmca_indicator", "usmca_rate_type_code", "usmca_ad_val_rate", 
        "usmca_specific_rate", "usmca_other_rate"
    ]
    cols_existentes = [c for c in columnas_interes if c in df.columns]
    df = df[cols_existentes].copy()

    # Filtrado estricto por indicadores S o S+ (TMEC aplicable)
    df['usmca_indicator'] = df['usmca_indicator'].str.strip().str.upper()
    df_filtrado = df[df['usmca_indicator'].isin(['S', 'S+'])].copy()

    print(f"   Registros filtrados (S o S+): {len(df_filtrado)}")
    print("   Calculando 'usmca_final_rate' aplicando reglas de formato...")
    
    # Invocamos la lógica estructural fila por fila de forma vectorizada
    if not df_filtrado.empty:
        df_filtrado['usmca_final_rate'] = df_filtrado.apply(__calcular_final_rate__, axis=1)

    return df_filtrado

### Fase 1: Procesamiento y Cálculo de Tarifas TMEC
Se inicializa el proceso invocando al orquestador. El sistema extraerá de la base cruda global exclusivamente los códigos elegibles para los beneficios del TMEC y decodificará sus tarifas.

In [3]:
DF_FRACCIONES_TMEC = procesar_tarifas_tmec(PATH_INPUT_TMEC)
print("\n>> ¡Extracción y decodificación de tarifas TMEC completada!")

### Fase 2: Exportación Final
El DataFrame filtrado y procesado se guarda en formato Excel en el directorio de salida intermedio, asegurando la creación de la ruta de carpetas de forma preventiva.

In [4]:
if not DF_FRACCIONES_TMEC.empty:
    print(f"Generando archivo Excel: {PATH_OUTPUT_TMEC}...")
    try:
        # Crear directorio padre por seguridad
        os.makedirs(os.path.dirname(PATH_OUTPUT_TMEC), exist_ok=True)
        
        # Exportar datos sin índice numérico
        DF_FRACCIONES_TMEC.to_excel(PATH_OUTPUT_TMEC, index=False)
        
        print("¡ÉXITO! Archivo consolidado correctamente.")
        print("\nVista previa de datos procesados:")
        print(DF_FRACCIONES_TMEC[['hts8', 'usmca_indicator', 'usmca_final_rate']].head())
    except PermissionError:
        print(f"❌ ERROR CRÍTICO: Asegúrate de tener cerrado el archivo '{PATH_OUTPUT_TMEC}' en Excel.")
    except Exception as e:
        print(f"❌ ERROR AL EXPORTAR: {e}")
else:
    print("⚠️ ADVERTENCIA: El resultado del procesamiento está vacío. No se generó ningún archivo.")